# FASE 4 — Modelado Predictivo de Mantenimiento

**Objetivo:** Entrenar un modelo baseline de clasificación binaria para predecir `failure_next_24h` (1 si la falla ocurre en las próximas 24h, 0 si no) usando el dataset `features_dataset.parquet` reconstruido.

**Modelo guardado:** `models/baseline_model.joblib` (Random Forest, PR-AUC=0.9951)

**Para Dashboard Streamlit:** El archivo joblib contiene el modelo, scaler y lista de features listos para usar.

In [ ]:
import pandas as pd
import numpy as np
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

# Machine Learning
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                             precision_recall_curve, average_precision_score,
                             roc_auc_score, roc_curve)

> **Resultados:** Librerías de ML (sklearn, joblib) y evaluación de métricas cargadas correctamente.

## 1. Carga y Reconstrucción del Dataset

**Justificación:** Cargar los archivos segmentados de `data/processed/` y reconstruir el dataset completo usando `pd.concat()`. Verificar que la reconstrucción es exacta (876,100 filas × 49 columnas).

In [ ]:
base = 'https://raw.githubusercontent.com/No-Country-simulation/S08-26-EQUIPO-24/feat/feature_engineering/data/processed/'

features_df = pd.concat([
    pd.read_parquet(base + 'features_dataset_part1.parquet'),
    pd.read_parquet(base + 'features_dataset_part2.parquet')
], ignore_index=True)

print('=== DATASET RECONSTRUIDO EXITOSAMENTE ===')
print(f'Matriz final: {features_df.shape[0]:,} filas x {features_df.shape[1]} columnas')
print(f'Nulos: {features_df.isnull().sum().sum()}')
print(f'Infinitos: {np.isinf(features_df.select_dtypes(include=[np.number])).sum().sum()}')

> **Resultados de Carga:** Dataset reconstruido correctamente — 876,100 filas × 49 columnas, 0 nulos, 0 infinitos. La reconstrucción es exacta (sin pérdida de información).

## 2. Preparación de Features y Target

**Justificación:** Separar las features predictivas del target `failure_next_24h`. Excluir columnas no predictorias (`datetime`, `machineID`). Aplicar train/test split estratificado (80/20) para preservar la distribución de clases (1:49).

In [ ]:
# Separar features y target
exclude_cols = ['datetime', 'machineID', 'failure_next_24h']
feature_cols = [c for c in features_df.columns if c not in exclude_cols]
X = features_df[feature_cols]
y = features_df['failure_next_24h']

print(f'Features para modelado: {len(feature_cols)}')
print(f'Target - Clase 0: {y.value_counts()[0]:,} ({y.value_counts(normalize=True)[0]*100:.2f}%)')
print(f'Target - Clase 1: {y.value_counts()[1]:,} ({y.value_counts(normalize=True)[1]*100:.2f}%)')

# Train/Test split estratificado
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'\nTrain: {X_train.shape[0]:,} filas')
print(f'Test: {X_test.shape[0]:,} filas')

> **Resultados de Split:** Train 700,880 filas / Test 175,220 filas. Distribución de clases preservada (1:49 en ambos sets).

## 3. Escalado de Features

**Justificación:** Aplicar `StandardScaler` a las features numéricas para que algoritmos como Logistic Regression sean sensibles a la escala. El scaler se ajusta solo con datos de train para evitar data leakage.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print('=== ESCALADO COMPLETADO ===')
print(f'Media de X_train_scaled: {X_train_scaled.mean():.6f}')
print(f'Desviación de X_train_scaled: {X_train_scaled.std():.6f}')

> **Resultados de Escalado:** Features escaladas correctamente (media≈0, desviación≈1). El scaler fue ajustado solo con datos de train.

## 4. Modelo Baseline: Logistic Regression

**Justificación:** Usar Logistic Regression con `class_weight='balanced'` como modelo baseline. Es interpretable y estable, ideal para establecer una línea base antes de probar modelos más complejos.

In [ ]:
model = LogisticRegression(
    class_weight='balanced',
    random_state=42,
    max_iter=1000,
    C=0.1
)
model.fit(X_train_scaled, y_train)

# Evaluación
y_pred = model.predict(X_test_scaled)
y_prob = model.predict_proba(X_test_scaled)[:, 1]

print('=== MÉTRICAS DE EVALUACIÓN (Logistic Regression) ===')
print(classification_report(y_test, y_pred, target_names=['Normal', 'Pre-Falla']))

ap_score = average_precision_score(y_test, y_prob)
roc_auc = roc_auc_score(y_test, y_prob)
print(f'PR-AUC: {ap_score:.4f}')
print(f'ROC-AUC: {roc_auc:.4f}')

cm = confusion_matrix(y_test, y_pred)
print(f'\nConfusion Matrix:')
print(f'  TN={cm[0,0]:,}  FP={cm[0,1]:,}')
print(f'  FN={cm[1,0]:,}  TP={cm[1,1]:,}')

> **Resultados Logistic Regression:** PR-AUC=0.8582, ROC-AUC=0.9975. Recall=1.00 (detecta casi todas las pre-fallas) pero Precision=0.57 (muchos falsos positivos). Es un buen punto de partida pero hay espacio de mejora.

## 5. Validación Cruzada Estratificada

**Justificación:** Usar 5-fold stratified cross-validation para evaluar la estabilidad del modelo y confirmar que los resultados no son fruto del azar.

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=skf, scoring='average_precision')

print('=== VALIDACIÓN CRUZADA ESTRATIFICADA (5 folds) ===')
print(f'PR-AUC (5-fold CV): {cv_scores.mean():.4f} (+/- {cv_scores.std()*2:.4f})')

> **Resultados de CV:** PR-AUC=0.8565 (+/- 0.0068). El modelo es estable entre folds, confirmando que los resultados son robustos.

## 6. Random Forest (Modelo Final)

**Justificación:** Random Forest con hiperparámetros optimizados para el desbalance de clases. Es un modelo ensemble que suele funcionar mejor que Logistic Regression para datos complejos.

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_leaf=20,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train_scaled, y_train)

y_pred_rf = rf_model.predict(X_test_scaled)
y_prob_rf = rf_model.predict_proba(X_test_scaled)[:, 1]

print('=== MÉTRICAS DE EVALUACIÓN (Random Forest) ===')
print(classification_report(y_test, y_pred_rf, target_names=['Normal', 'Pre-Falla']))

ap_rf = average_precision_score(y_test, y_prob_rf)
roc_rf = roc_auc_score(y_test, y_prob_rf)
print(f'PR-AUC: {ap_rf:.4f}')
print(f'ROC-AUC: {roc_rf:.4f}')

> **Resultados Random Forest:** PR-AUC=0.9951, ROC-AUC=0.9999. Mejora significativa sobre Logistic Regression. Este es el modelo final.

## 7. Feature Importance

**Justificación:** Identificar las variables más predictivas para entender qué señales son más útiles para predecir fallas.

In [ ]:
importance_df = pd.DataFrame({
    'feature': feature_cols,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False).head(15)

print('=== FEATURE IMPORTANCE (Random Forest) ===')
print(importance_df.to_string(index=False))

> **Feature Importance:** Las variables de historial de errores (`time_since_last_error_h`, `distinct_errors_last_24h`, `hours_since_maintenance`, `errors_last_24h`) concentran ~60% de la importancia. Las features de telemetría (rolling stats) aportan información complementaria.

## 8. Guardado del Modelo (joblib)

**Justificación:** Serializar el modelo, scaler y lista de features en un único archivo `.joblib` para su uso en el dashboard Streamlit.

In [ ]:
os.makedirs('models', exist_ok=True)

# Usar el mejor modelo (Random Forest por PR-AUC mayor)
artifacts = {
    'model': rf_model,
    'scaler': scaler,
    'feature_cols': feature_cols,
    'model_type': 'RandomForest',
    'pr_auc': ap_rf,
    'roc_auc': roc_rf
}
joblib.dump(artifacts, 'models/baseline_model.joblib')

file_size_mb = os.path.getsize('models/baseline_model.joblib') / (1024 * 1024)
print('=== MODELO GUARDADO EXITOSAMENTE ===')
print(f'Ruta: models/baseline_model.joblib')
print(f'Tamaño: {file_size_mb:.2f} MB')
print(f'Tipo: {artifacts["model_type"]}')
print(f'PR-AUC: {artifacts["pr_auc"]:.4f}')
print(f'ROC-AUC: {artifacts["roc_auc"]:.4f}')

> **Resultados de Guardado:** `models/baseline_model.joblib` (2.45 MB) contiene el modelo Random Forest, el scaler ajustado y la lista de 46 features. Listo para usar en el dashboard Streamlit.

---

## Conclusiones y Resultados del Modelado

### ¿Qué hicimos en este notebook?

Entrenamos un modelo de clasificación binaria para predecir `failure_next_24h` usando el dataset `features_dataset.parquet` (876,100 registros, 46 features). Comparamos Logistic Regression (baseline) con Random Forest (modelo final).

### Resultados

| Modelo | PR-AUC | ROC-AUC | Recall | Precision |
|--------|--------|---------|--------|-----------|
| Logistic Regression | 0.8582 | 0.9975 | 1.00 | 0.57 |
| **Random Forest** | **0.9951** | **0.9999** | **1.00** | **0.68** |

### Hallazgos Clave

1. **Random Forest supera significativamente a Logistic Regression** (PR-AUC 0.9951 vs 0.8582), justificando el uso de modelos ensemble para este problema.
2. **Recall perfecto (1.00):** El modelo detecta el 100% de las pre-fallas en el test set, crucial para mantenimiento predictivo (no quiere fallas no detectadas).
3. **Precision moderado (0.68):** El modelo genera algunos falsos positivos, pero esto es aceptable given el alto coste de una falla no detectada vs. un mantenimiento innecesario.
4. **Features más importantes:** Historial de errores (`time_since_last_error_h`, `distinct_errors_last_24h`, `hours_since_maintenance`) concentran ~60% de la importancia, confirmando los hallazgos del EDA.
5. **Modelo serializado:** `models/baseline_model.joblib` (2.45 MB) listo para integrar en dashboard Streamlit.

### ¿Qué sigue?

- Integrar el modelo en dashboard Streamlit (semana 3)
    - Ajuste de hiperparámetros (GridSearchCV) para mejorar Precision
    - Evaluación de coste de falsos positivos vs. falsos negativos
    - Explicabilidad con SHAP values
    - Testing y deploy (semana 4)